In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# --- CNN ---
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader
    USE_CNN = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🧠 PyTorch (device: {device})")
except ImportError:
    USE_CNN = False
    print("⚠️ PyTorchなし → CNNスキップ")

# ============================================================
# 1. データ読み込み
# ============================================================
print("📂 データ読み込み中...")
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

# ============================================================
# 2. 波数・バンド定義（水バンドのみ — 樹種依存を排除）
# ============================================================
wavenumbers = np.array([float(c) for c in spec_cols])

# 水バンドのインデックス（面積計算用）
band_water_5150 = np.where((wavenumbers >= 5000) & (wavenumbers <= 5300))[0]
band_water_6900 = np.where((wavenumbers >= 6700) & (wavenumbers <= 7100))[0]

print(f"📏 スペクトル次元数: {len(spec_cols)}")
print(f"  Band [water_5150]: {len(band_water_5150)} points "
      f"({wavenumbers[band_water_5150[0]]:.0f}–{wavenumbers[band_water_5150[-1]]:.0f})")
print(f"  Band [water_6900]: {len(band_water_6900)} points "
      f"({wavenumbers[band_water_6900[0]]:.0f}–{wavenumbers[band_water_6900[-1]]:.0f})")


# ============================================================
# 3. 前処理関数
# ============================================================

def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s

def apply_derivatives(X):
    d1 = savgol_filter(X, window_length=15, polyorder=2, deriv=1, axis=1)
    d2 = savgol_filter(X, window_length=11, polyorder=2, deriv=2, axis=1)
    return d1, d2


# ============================================================
# 4. 物理特徴量（厳選8個：樹種非依存のみ）
# ============================================================

def extract_features_minimal(X_raw, X_d2):
    """
    樹種に依存しない、普遍的な物理特徴量のみを抽出
    セルロース/リグニン個別バンドは意図的に除外
    """
    f = {}

    # --- 罠B対策: 密度・散乱プロキシ（SNVで消える情報） ---
    f['raw_mean'] = np.mean(X_raw, axis=1)
    f['raw_std']  = np.std(X_raw, axis=1)

    # --- 罠1対策: 水バンド面積（点→面のロバスト化）---
    d2_w5150 = np.abs(X_d2[:, band_water_5150])
    d2_w6900 = np.abs(X_d2[:, band_water_6900])

    f['area_water_5150'] = np.trapezoid(d2_w5150, axis=1)
    f['area_water_6900'] = np.trapezoid(d2_w6900, axis=1)

    # --- 罠3対策: ピークシフト検出 ---
    f['peak_wn_5150'] = wavenumbers[band_water_5150][
        np.argmax(d2_w5150, axis=1)
    ]
    f['peak_wn_6900'] = wavenumbers[band_water_6900][
        np.argmax(d2_w6900, axis=1)
    ]

    # --- 樹種非依存の比率 ---
    # 水量 / 散乱量 → 密度で正規化した含水率指標
    f['ratio_water_scatter'] = f['area_water_5150'] / (f['raw_mean'] + 1e-8)
    # 2帯域比 → 自由水/結合水のバランス（物理的に普遍）
    f['ratio_5150_6900'] = f['area_water_5150'] / (f['area_water_6900'] + 1e-8)

    return pd.DataFrame(f)


# ============================================================
# 5. CNN定義（元コードと同じ）
# ============================================================
if USE_CNN:
    class NIR_CNN(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.conv_block = nn.Sequential(
                nn.Conv1d(1, 32, kernel_size=15, padding=7),
                nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(4), nn.Dropout(0.2),
                nn.Conv1d(32, 64, kernel_size=11, padding=5),
                nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(4), nn.Dropout(0.2),
                nn.Conv1d(64, 128, kernel_size=7, padding=3),
                nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(8), nn.Dropout(0.3),
            )
            self.fc = nn.Sequential(
                nn.Linear(128 * 8, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 1)
            )
        def forward(self, x):
            x = x.unsqueeze(1)
            x = self.conv_block(x)
            x = x.view(x.size(0), -1)
            return self.fc(x).squeeze(-1)

    def train_cnn(X_tr, y_tr, X_va, y_va, input_dim, epochs=200, lr=1e-3):
        model = NIR_CNN(input_dim).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        criterion = nn.MSELoss()
        ds = TensorDataset(torch.FloatTensor(X_tr).to(device),
                           torch.FloatTensor(y_tr).to(device))
        loader = DataLoader(ds, batch_size=64, shuffle=True)
        best_loss, best_state, patience = float('inf'), None, 0
        for _ in range(epochs):
            model.train()
            for xb, yb in loader:
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()
            scheduler.step()
            model.eval()
            with torch.no_grad():
                vl = criterion(model(torch.FloatTensor(X_va).to(device)),
                               torch.FloatTensor(y_va).to(device)).item()
            if vl < best_loss:
                best_loss = vl
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= 20:
                    break
        model.load_state_dict(best_state)
        model.eval()
        return model


# ============================================================
# 6. CVループ
# ============================================================
print("\n" + "=" * 60)
print("🚀 厳選改善モデル — CVループ開始")
print("=" * 60)

gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

final_lgb = np.zeros(len(test))
final_pls = np.zeros(len(test))
final_rdg = np.zeros(len(test))
final_cnn = np.zeros(len(test)) if USE_CNN else None

oof_lgb = np.zeros(len(train))
oof_pls = np.zeros(len(train))
oof_rdg = np.zeros(len(train))
oof_cnn = np.zeros(len(train)) if USE_CNN else None

fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(
    train[spec_cols].values, y_train_log, groups
)):
    print(f"\n{'─'*55}")
    print(f"📁 Fold {fold+1}/5  (train: {len(tr_idx)}, valid: {len(va_idx)})")
    va_species = train.iloc[va_idx]['樹種'].unique()
    print(f"   検証樹種: {list(va_species)}")
    print(f"{'─'*55}")

    X_tr_raw = train[spec_cols].values[tr_idx]
    y_tr = y_train_log.iloc[tr_idx].values
    X_va_raw = train[spec_cols].values[va_idx]
    y_va = y_train_log.iloc[va_idx].values
    X_te_raw = X_test_raw.copy()

    # ── 前処理 ──
    snv_tr = apply_snv(X_tr_raw)
    snv_va = apply_snv(X_va_raw)
    snv_te = apply_snv(X_te_raw)

    d1_tr, d2_tr = apply_derivatives(snv_tr)
    d1_va, d2_va = apply_derivatives(snv_va)
    d1_te, d2_te = apply_derivatives(snv_te)

    # ── 物理特徴量（厳選8個）──
    phys_tr = extract_features_minimal(X_tr_raw, d2_tr)
    phys_va = extract_features_minimal(X_va_raw, d2_va)
    phys_te = extract_features_minimal(X_te_raw, d2_te)

    # ── PCA（KNN用=5次元、LGB用=10次元：元コード寄り）──
    pca_knn = PCA(n_components=5, random_state=42)
    pca_knn_tr = pca_knn.fit_transform(snv_tr)
    pca_knn_va = pca_knn.transform(snv_va)
    pca_knn_te = pca_knn.transform(snv_te)

    pca_lgb = PCA(n_components=10, random_state=42)
    pca_lgb_tr = pca_lgb.fit_transform(snv_tr)
    pca_lgb_va = pca_lgb.transform(snv_va)
    pca_lgb_te = pca_lgb.transform(snv_te)

    # ── KNN特徴量（元コード寄り：k=5、PCA-5空間）──
    knn = NearestNeighbors(n_neighbors=5, metric='cosine')
    knn.fit(pca_knn_tr)

    dist_tr, ind_tr = knn.kneighbors(pca_knn_tr, n_neighbors=6)
    knn_ymean_tr = np.mean(y_tr[ind_tr[:, 1:]], axis=1).reshape(-1, 1)

    dist_va, ind_va = knn.kneighbors(pca_knn_va, n_neighbors=5)
    knn_ymean_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)

    dist_te, ind_te = knn.kneighbors(pca_knn_te, n_neighbors=5)
    knn_ymean_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

    # ── 特徴量の次元チェック ──
    if fold == 0:
        n_snv = snv_tr.shape[1]
        n_d1 = d1_tr.shape[1]
        n_pca = pca_lgb_tr.shape[1]
        n_knn = knn_ymean_tr.shape[1]
        n_phys = phys_tr.shape[1]
        total_lgb = n_snv + n_d1 + n_pca + n_knn + n_phys
        total_rdg = n_pca + n_knn + n_phys

        print(f"\n  📐 特徴量次元チェック:")
        print(f"     SNV:         {n_snv}")
        print(f"     d1:          {n_d1}")
        print(f"     PCA(LGB用):  {n_pca}")
        print(f"     PCA(KNN用):  {pca_knn_tr.shape[1]}")
        print(f"     KNN:         {n_knn}")
        print(f"     物理特徴量:  {n_phys}  {list(phys_tr.columns)}")
        print(f"     ──────────────────")
        print(f"     LGB合計:     {total_lgb}")
        print(f"     Ridge合計:   {total_rdg}")
        print(f"     PLS入力:     {d2_tr.shape[1]} (d2)")
        if USE_CNN:
            print(f"     CNN入力:     {snv_tr.shape[1]} (SNV)")

    # ─────────────────────────────────────
    # モデル1: LightGBM
    # ─────────────────────────────────────
    feat_tr_lgb = np.hstack([snv_tr, d1_tr, pca_lgb_tr, knn_ymean_tr, phys_tr.values])
    feat_va_lgb = np.hstack([snv_va, d1_va, pca_lgb_va, knn_ymean_va, phys_va.values])
    feat_te_lgb = np.hstack([snv_te, d1_te, pca_lgb_te, knn_ymean_te, phys_te.values])

    lgb_model = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.03, max_depth=5,
        num_leaves=31, subsample=0.8, colsample_bytree=0.3,
        min_child_samples=20, reg_alpha=0.0, reg_lambda=0.0,
        random_state=42, verbosity=-1
    )
    lgb_model.fit(
        feat_tr_lgb, y_tr,
        eval_set=[(feat_va_lgb, y_va)],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    p_va_lgb = np.expm1(lgb_model.predict(feat_va_lgb))
    p_te_lgb = np.expm1(lgb_model.predict(feat_te_lgb))

    # ─────────────────────────────────────
    # モデル2: PLS回帰（複数n_components平均で安定化）
    # ─────────────────────────────────────
    pls_va_list, pls_te_list = [], []
    for nc in [5, 7, 9]:
        pls_m = PLSRegression(n_components=nc)
        pls_m.fit(d2_tr, y_tr)
        pls_va_list.append(np.expm1(pls_m.predict(d2_va).flatten()))
        pls_te_list.append(np.expm1(pls_m.predict(d2_te).flatten()))
    p_va_pls = np.mean(pls_va_list, axis=0)
    p_te_pls = np.mean(pls_te_list, axis=0)

    # ─────────────────────────────────────
    # モデル3: Ridge回帰
    # ─────────────────────────────────────
    feat_tr_rdg = np.hstack([pca_lgb_tr, knn_ymean_tr, phys_tr.values])
    feat_va_rdg = np.hstack([pca_lgb_va, knn_ymean_va, phys_va.values])
    feat_te_rdg = np.hstack([pca_lgb_te, knn_ymean_te, phys_te.values])

    # 外れ値クリップ + スケーリング
    lo = np.percentile(feat_tr_rdg, 1, axis=0)
    hi = np.percentile(feat_tr_rdg, 99, axis=0)
    feat_tr_rdg = np.clip(feat_tr_rdg, lo, hi)
    feat_va_rdg = np.clip(feat_va_rdg, lo, hi)
    feat_te_rdg = np.clip(feat_te_rdg, lo, hi)

    scaler_rdg = StandardScaler()
    feat_tr_rdg_s = scaler_rdg.fit_transform(feat_tr_rdg)
    feat_va_rdg_s = scaler_rdg.transform(feat_va_rdg)
    feat_te_rdg_s = scaler_rdg.transform(feat_te_rdg)

    rdg_model = Ridge(alpha=10.0)
    rdg_model.fit(feat_tr_rdg_s, y_tr)
    p_va_rdg = np.expm1(np.clip(rdg_model.predict(feat_va_rdg_s), 0, 6.5))
    p_te_rdg = np.expm1(np.clip(rdg_model.predict(feat_te_rdg_s), 0, 6.5))

    # ─────────────────────────────────────
    # モデル4: CNN
    # ─────────────────────────────────────
    if USE_CNN:
        sc_cnn = StandardScaler()
        cnn_tr = sc_cnn.fit_transform(snv_tr)
        cnn_va = sc_cnn.transform(snv_va)
        cnn_te = sc_cnn.transform(snv_te)
        cnn_model = train_cnn(cnn_tr, y_tr, cnn_va, y_va,
                              input_dim=cnn_tr.shape[1], epochs=200, lr=1e-3)
        with torch.no_grad():
            p_va_cnn = np.expm1(
                cnn_model(torch.FloatTensor(cnn_va).to(device)).cpu().numpy())
            p_te_cnn = np.expm1(
                cnn_model(torch.FloatTensor(cnn_te).to(device)).cpu().numpy())

    # ─────────────────────────────────────
    # ブレンド（元コード寄りの手動重み）
    # ─────────────────────────────────────
    if USE_CNN:
        w_lgb, w_pls, w_rdg, w_cnn = 0.55, 0.20, 0.10, 0.15
        p_va_blend = (p_va_lgb * w_lgb + p_va_pls * w_pls
                      + p_va_rdg * w_rdg + p_va_cnn * w_cnn)
    else:
        w_lgb, w_pls, w_rdg = 0.60, 0.25, 0.15
        p_va_blend = p_va_lgb * w_lgb + p_va_pls * w_pls + p_va_rdg * w_rdg

    # OOF蓄積
    oof_lgb[va_idx] = p_va_lgb
    oof_pls[va_idx] = p_va_pls
    oof_rdg[va_idx] = p_va_rdg
    if USE_CNN:
        oof_cnn[va_idx] = p_va_cnn

    final_lgb += p_te_lgb / 5
    final_pls += p_te_pls / 5
    final_rdg += p_te_rdg / 5
    if USE_CNN:
        final_cnn += p_te_cnn / 5

    # Fold RMSE表示
    y_va_real = np.expm1(y_va)
    rmse_lgb = np.sqrt(mean_squared_error(y_va_real, p_va_lgb))
    rmse_pls = np.sqrt(mean_squared_error(y_va_real, p_va_pls))
    rmse_rdg = np.sqrt(mean_squared_error(y_va_real, p_va_rdg))
    rmse_blend = np.sqrt(mean_squared_error(y_va_real, p_va_blend))
    fold_rmses.append(rmse_blend)

    print(f"  LGB        RMSE: {rmse_lgb:.4f}")
    print(f"  PLS(avg)   RMSE: {rmse_pls:.4f}")
    print(f"  Ridge      RMSE: {rmse_rdg:.4f}")
    if USE_CNN:
        rmse_cnn = np.sqrt(mean_squared_error(y_va_real, p_va_cnn))
        print(f"  CNN        RMSE: {rmse_cnn:.4f}")
    print(f"  🌟 Blend   RMSE: {rmse_blend:.4f}")


# ============================================================
# 7. 全体OOF評価
# ============================================================
print(f"\n{'='*60}")
print("📊 全体OOF評価")
print(f"{'='*60}")

y_true_real = np.expm1(y_train_log)

# 各モデル単体のOOF
rmse_oof_lgb = np.sqrt(mean_squared_error(y_true_real, oof_lgb))
rmse_oof_pls = np.sqrt(mean_squared_error(y_true_real, oof_pls))
rmse_oof_rdg = np.sqrt(mean_squared_error(y_true_real, oof_rdg))
print(f"  LGB      OOF RMSE: {rmse_oof_lgb:.4f}")
print(f"  PLS(avg) OOF RMSE: {rmse_oof_pls:.4f}")
print(f"  Ridge    OOF RMSE: {rmse_oof_rdg:.4f}")
if USE_CNN:
    rmse_oof_cnn = np.sqrt(mean_squared_error(y_true_real, oof_cnn))
    print(f"  CNN      OOF RMSE: {rmse_oof_cnn:.4f}")

# 手動ブレンドのOOF
if USE_CNN:
    oof_blend = (oof_lgb * w_lgb + oof_pls * w_pls
                 + oof_rdg * w_rdg + oof_cnn * w_cnn)
else:
    oof_blend = oof_lgb * w_lgb + oof_pls * w_pls + oof_rdg * w_rdg

oof_rmse = np.sqrt(mean_squared_error(y_true_real, oof_blend))
print(f"\n  🌟 Blend OOF RMSE: {oof_rmse:.4f}")
print(f"  📊 Fold平均 RMSE:  {np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")

# ── 参考: 複数の手動重みでの比較 ──
print(f"\n  --- 重み感度分析 ---")
weight_sets = {
    "LGB重視   (0.70/0.15/0.15)": [0.70, 0.15, 0.15, 0.00],
    "LGB+PLS   (0.55/0.30/0.15)": [0.55, 0.30, 0.15, 0.00],
    "元コード風 (0.55/0.20/0.10)": [0.55, 0.20, 0.10, 0.15],
    "PLS重視   (0.30/0.40/0.10)": [0.30, 0.40, 0.10, 0.20],
    "LGB単独":                     [1.00, 0.00, 0.00, 0.00],
    "PLS単独":                     [0.00, 1.00, 0.00, 0.00],
}
for name, w in weight_sets.items():
    if USE_CNN:
        b = w[0]*oof_lgb + w[1]*oof_pls + w[2]*oof_rdg + w[3]*oof_cnn
    else:
        ws = w[0] + w[1] + w[2]
        if ws > 0:
            b = (w[0]*oof_lgb + w[1]*oof_pls + w[2]*oof_rdg) / ws
        else:
            b = oof_lgb
    r = np.sqrt(mean_squared_error(y_true_real, b))
    print(f"    {name:35s} OOF RMSE: {r:.4f}")


# ============================================================
# 8. 提出ファイル
# ============================================================
if USE_CNN:
    final_blend = (final_lgb * w_lgb + final_pls * w_pls
                   + final_rdg * w_rdg + final_cnn * w_cnn)
else:
    final_blend = final_lgb * w_lgb + final_pls * w_pls + final_rdg * w_rdg

final_blend = np.clip(final_blend, 0, None)

submit[1] = final_blend
output_filename = 'submission_selective_improve.csv'
submit.to_csv(output_filename, index=False, header=False)

print(f"\n✅ 提出ファイル: {output_filename}")
print(f"📈 予測統計: min={final_blend.min():.1f}%, "
      f"median={np.median(final_blend):.1f}%, max={final_blend.max():.1f}%")

🧠 PyTorch (device: cuda)
📂 データ読み込み中...
📏 スペクトル次元数: 1555
  Band [water_5150]: 78 points (5300–5003)
  Band [water_6900]: 103 points (7097–6704)

🚀 厳選改善モデル — CVループ開始

───────────────────────────────────────────────────────
📁 Fold 1/5  (train: 940, valid: 270)
   検証樹種: ['ウエンジ', 'トチ']
───────────────────────────────────────────────────────

  📐 特徴量次元チェック:
     SNV:         1555
     d1:          1555
     PCA(LGB用):  10
     PCA(KNN用):  5
     KNN:         1
     物理特徴量:  8  ['raw_mean', 'raw_std', 'area_water_5150', 'area_water_6900', 'peak_wn_5150', 'peak_wn_6900', 'ratio_water_scatter', 'ratio_5150_6900']
     ──────────────────
     LGB合計:     3129
     Ridge合計:   19
     PLS入力:     1555 (d2)
     CNN入力:     1555 (SNV)
  LGB        RMSE: 12.1489
  PLS(avg)   RMSE: 10.8016
  Ridge      RMSE: 17.3470
  CNN        RMSE: 11.1059
  🌟 Blend   RMSE: 10.6436

───────────────────────────────────────────────────────
📁 Fold 2/5  (train: 981, valid: 229)
   検証樹種: ['チェリー', 'ヒノキ']
───────────────────